<a href="https://colab.research.google.com/github/Ghhadaa/project-try/blob/main/Yet_another_copy_of_locknew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""Untitled1.ipynb

Automatically generated by Colab.
"""

import os
import time
import math
import re
from pathlib import Path
from collections import Counter
from itertools import islice
import requests
import statistics
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock   # 🔒 لاستخدام القفل

# =====================================================
# GLOBALS (بعد إصلاح الريس بالـ Lock)
# =====================================================
GLOBAL_TOTAL_TOKENS = 0
GLOBAL_BOOK_COUNTER = 0
GLOBAL_TFIDF_SUM = 0.0

lock = Lock()   # 🔒 مثل ما قالت الدكتورة
# =====================================================

# Performance bookkeeping
_stage_times = {}

def time_stage(name):
    def deco(fn):
        def wrapper(*args, **kwargs):
            t0 = time.perf_counter()
            print(f"[stage start] {name} ...")
            res = fn(*args, **kwargs)
            elapsed = time.perf_counter() - t0
            _stage_times[name] = elapsed
            print(f"[stage done]  {name} — {elapsed:.2f}s")
            return res
        return wrapper
    return deco

def print_performance_report():
    total = sum(_stage_times.values())
    print("\n=== PERFORMANCE REPORT (per stage) ===")
    for k, v in _stage_times.items():
        print(f"  {k:30s} : {v:7.2f}s")
    print(f"  {'TOTAL (sum stages)':30s} : {total:7.2f}s")
    print("======================================\n")

# --- Config & books ---
DATA_DIR = Path("gutenberg_books")
DATA_DIR.mkdir(exist_ok=True)
ENCODING = "utf-8"
GUTENBERG_BOOKS = {
    "pride_and_prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "moby_dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "war_and_peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    "ulysses": "https://www.gutenberg.org/files/4300/4300-0.txt",
    "great_expectations": "https://www.gutenberg.org/files/1400/1400-0.txt",
    "alice_in_wonderland": "https://www.gutenberg.org/files/11/11-0.txt",
    "sherlock_holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
}

# --- spaCy / NLTK / WordNet / tqdm / matplotlib ---
try:
    import spacy
except Exception:
    print("Installing spaCy...")
    get_ipython().system("pip install -q spacy")
    import spacy

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except Exception:
    print("Installing spaCy english model (en_core_web_sm)...")
    get_ipython().system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

try:
    import nltk
except Exception:
    print("Installing nltk...")
    get_ipython().system("pip install -q nltk")
    import nltk

import nltk
try:
    from nltk.corpus import wordnet as wn
    wn.synsets("test")
except Exception:
    print("Downloading WordNet corpus for NLTK...")
    nltk.download('wordnet')
    from nltk.corpus import wordnet as wn

# 🔹 PRELOAD WORDNET لتقليل أخطاء NLTK الداخلية
print("Preloading WordNet before threading...")
_ = wn.synsets("dog")
_ = wn.synsets("run")
_ = wn.synsets("book")
print("WordNet is ready.\n")

try:
    from tqdm import tqdm
except Exception:
    get_ipython().system("pip install -q tqdm")
    from tqdm import tqdm

try:
    import matplotlib.pyplot as plt
except Exception:
    get_ipython().system("pip install -q matplotlib")
    import matplotlib.pyplot as plt

WORD_RE = re.compile(r"^[A-Za-z']+$")
ALLOWED_POS = {"NOUN", "VERB", "ADJ", "ADV"}
MIN_LEMMA_LEN = 3

# --------------------------------------------------
# 1) DOWNLOAD BOOKS
# --------------------------------------------------

def _download_one_book(name, url, out_dir):
    dest = out_dir / f"{name}.txt"
    if dest.exists():
        print(f"[skip] {name} already downloaded.")
        return name, dest
    print(f"[download] {name} ...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dest.write_text(r.text, encoding=ENCODING)
    print(f"[saved] {dest} ({dest.stat().st_size/1024:.1f} KB)")
    return name, dest

@time_stage("download_books")
def download_books(book_map, out_dir=DATA_DIR):
    saved = {}
    max_workers = min(len(book_map), os.cpu_count() or 4)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(_download_one_book, name, url, out_dir): name
            for name, url in book_map.items()
        }
        for fut in as_completed(futures):
            name, dest = fut.result()
            saved[name] = dest
    return saved

# --------------------------------------------------
# 2) READ & CLEAN
# --------------------------------------------------

def _read_and_clean_one(name, path):
    text = path.read_text(encoding=ENCODING, errors="ignore")
    start_m = re.search(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG", text, re.IGNORECASE)
    end_m = re.search(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG", text, re.IGNORECASE)
    if start_m:
        text = text[start_m.end():]
    if end_m:
        text = text[:end_m.start()]
    text = re.sub(r"End of Project Gutenberg.*", "", text, flags=re.IGNORECASE)
    return name, text, len(text)

@time_stage("read_and_clean")
def read_and_clean(files_map):
    texts = {}
    max_workers = min(len(files_map), os.cpu_count() or 4)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(_read_and_clean_one, name, path): name
            for name, path in files_map.items()
        }
        for fut in as_completed(futures):
            name, text, length = fut.result()
            texts[name] = text
            print(f"  read: {name} — {length/1024:.1f} KB")
    return texts

# --------------------------------------------------
# 3) POS + LEMMAS
# --------------------------------------------------

def _extract_lemmas_one(name, text, batch_size):
    lemmas = []
    for i in range(0, len(text), batch_size):
        chunk = text[i:i+batch_size]
        if not chunk.strip():
            continue
        doc = nlp(chunk)
        for tok in doc:
            if tok.is_space or tok.is_punct:
                continue
            pos = tok.pos_
            if pos not in ALLOWED_POS:
                continue
            lemma = tok.lemma_.lower().strip()
            if len(lemma) < MIN_LEMMA_LEN:
                continue
            if tok.is_stop:
                continue
            if not WORD_RE.fullmatch(lemma):
                continue

            # حماية من WordNetError / ValueError
            try:
                has_synsets = len(wn.synsets(lemma)) > 0
            except Exception:
                has_synsets = False

            if not has_synsets:
                if lemma.endswith("'s"):
                    lemma2 = lemma[:-2]
                    try:
                        if len(wn.synsets(lemma2)) == 0:
                            continue
                    except Exception:
                        continue
                else:
                    continue

            lemmas.append(lemma)
    return name, lemmas

@time_stage("pos_lemmatize_meaningful")
def extract_meaningful_lemmas_per_book(texts, batch_size=25000):
    per_book_lemmas = {}
    max_workers = min(len(texts), os.cpu_count() or 4)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(_extract_lemmas_one, name, text, batch_size): name
            for name, text in texts.items()
        }
        for fut in as_completed(futures):
            name, lemmas = fut.result()
            per_book_lemmas[name] = lemmas
            print(f"  extracted meaningful lemmas for {name}: tokens={len(lemmas):,}, unique={len(set(lemmas)):,}")
    return per_book_lemmas

# --------------------------------------------------
# 4) N-GRAMS  (مع Lock على القيم القلوبل)
# --------------------------------------------------

def ngrams_from_list(tokens, n):
    if n == 1:
        for t in tokens:
            yield (t,)
    else:
        for i in range(len(tokens) - n + 1):
            yield tuple(tokens[i:i+n])

@time_stage("ngram_counting")
def compute_ngrams(per_book_lemmas):

    agg_uni = Counter()
    agg_bi = Counter()
    agg_tri = Counter()
    per_book_uni = {}
    per_book_token_counts = {}

    def _process_book(name, lemmas):
        global GLOBAL_TOTAL_TOKENS, GLOBAL_BOOK_COUNTER

        nonlocal agg_uni, agg_bi, agg_tri, per_book_uni, per_book_token_counts
        uni = Counter(lemmas)
        bi = Counter(ngrams_from_list(lemmas, 2))
        tri = Counter(ngrams_from_list(lemmas, 3))

        agg_uni.update(uni)
        agg_bi.update(bi)
        agg_tri.update(tri)
        per_book_uni[name] = uni
        per_book_token_counts[name] = len(lemmas)

        # 🔒 أسلوب الدكتورة: acquire / release
        lock.acquire()
        try:
            GLOBAL_TOTAL_TOKENS += len(lemmas)
            GLOBAL_BOOK_COUNTER += 1
        finally:
            lock.release()

        print(f"  ngrams for {name}: tokens={len(lemmas):,}, unique_unis={len(uni):,}, bigrams={len(bi):,}")

    max_workers = min(len(per_book_lemmas), os.cpu_count() or 4)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [
            ex.submit(_process_book, name, lemmas)
            for name, lemmas in per_book_lemmas.items()
        ]
        for fut in as_completed(futures):
            fut.result()

    total_tokens = sum(per_book_token_counts.values())

    return {
        "agg_uni": agg_uni,
        "agg_bi": agg_bi,
        "agg_tri": agg_tri,
        "per_book_uni": per_book_uni,
        "per_book_token_counts": per_book_token_counts,
        "total_tokens": total_tokens
    }

# --------------------------------------------------
# 5) TF-IDF  (Lock على GLOBAL_TFIDF_SUM)
# --------------------------------------------------

@time_stage("tfidf_computation")
def compute_tfidf(per_book_uni, per_book_token_counts, top_k_terms=1000):

    df = Counter()
    for cnts in per_book_uni.values():
        for term in cnts:
            df[term] += 1
    num_docs = len(per_book_uni)

    global_freq = Counter()
    for cnts in per_book_uni.values():
        global_freq.update(cnts)

    candidates = [w for w, _ in global_freq.most_common(top_k_terms)]

    tfidf_per_book = {}
    global_tfidf_scores = Counter()

    def _compute_for_book(name, cnts):
        global GLOBAL_TFIDF_SUM
        nonlocal tfidf_per_book, global_tfidf_scores

        doc_len = per_book_token_counts[name]
        scored = []
        for term in candidates:
            tf = cnts.get(term, 0) / max(1, doc_len)
            idf = math.log((1 + num_docs) / (1 + df.get(term, 0))) + 1
            score = tf * idf
            scored.append((term, score))

            global_tfidf_scores[term] += score

            # 🔒 Lock على الجمع
            lock.acquire()
            try:
                GLOBAL_TFIDF_SUM += score
            finally:
                lock.release()

        scored.sort(key=lambda x: x[1], reverse=True)
        tfidf_per_book[name] = scored[:50]
        print(f"  tfidf computed for {name}")

    max_workers = min(len(per_book_uni), os.cpu_count() or 4)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [
            ex.submit(_compute_for_book, name, cnts)
            for name, cnts in per_book_uni.items()
        ]
        for fut in as_completed(futures):
            fut.result()

    global_top = [t for t, _ in global_tfidf_scores.most_common(200)]
    return tfidf_per_book, global_top

# --------------------------------------------------
# 6) ZIPF + LEXICAL STATS
# --------------------------------------------------

@time_stage("plot_zipf")
def save_zipf_plot(agg_uni, filename="zipf_meaningful.png"):
    freqs = [f for _, f in agg_uni.most_common()]
    plt.figure(figsize=(7, 5))
    plt.loglog(freqs)
    plt.xlabel("Rank (log)")
    plt.ylabel("Frequency (log)")
    plt.title("Zipf Plot (meaningful lemmas)")
    plt.grid(True)
    plt.savefig(filename, bbox_inches="tight")
    plt.close()
    return filename

@time_stage("lexical_stats")
def compute_lexical_stats(agg_uni, total_tokens):
    unique_tokens = len(agg_uni)
    hapax = sum(1 for v in agg_uni.values() if v == 1)
    type_token_ratio = unique_tokens / max(1, total_tokens)

    def shannon(counter):
        tot = sum(counter.values())
        ent = 0.0
        for c in counter.values():
            p = c / tot
            ent -= p * math.log2(p)
        return ent

    entropy = shannon(agg_uni)
    return {
        "unique_tokens": unique_tokens,
        "hapax": hapax,
        "ttr": type_token_ratio,
        "entropy": entropy
    }

# --------------------------------------------------
# 7) PIPELINE & PROFILING
# --------------------------------------------------

def sequential_pipeline_once(run_download=True):
    _stage_times.clear()
    files_map = download_books(GUTENBERG_BOOKS) if run_download else {k: DATA_DIR / f"{k}.txt" for k in GUTENBERG_BOOKS}
    texts = read_and_clean(files_map)
    per_book_lemmas = extract_meaningful_lemmas_per_book(texts, batch_size=20000)
    ngram_results = compute_ngrams(per_book_lemmas)
    agg_uni = ngram_results["agg_uni"]
    agg_bi = ngram_results["agg_bi"]
    agg_tri = ngram_results["agg_tri"]
    per_book_uni = ngram_results["per_book_uni"]
    per_book_token_counts = ngram_results["per_book_token_counts"]
    total_tokens = ngram_results["total_tokens"]
    tfidf_per_book, global_top = compute_tfidf(per_book_uni, per_book_token_counts, top_k_terms=800)
    stats = compute_lexical_stats(agg_uni, total_tokens)
    zipf_path = save_zipf_plot(agg_uni, filename="zipf_meaningful.png")
    return {
        "agg_uni": agg_uni,
        "agg_bi": agg_bi,
        "agg_tri": agg_tri,
        "tfidf_per_book": tfidf_per_book,
        "global_top_candidates": global_top,
        "stats": stats,
        "zipf": zipf_path,
        "total_tokens": total_tokens,
        "per_stage_times": dict(_stage_times)
    }

def run_timeit_on_sequential(repeats=3, number=1):
    import timeit
    setup = "from __main__ import sequential_pipeline_once"
    stmt = "sequential_pipeline_once(run_download=False)"
    print(f"Running timeit.repeat (repeats={repeats}, number={number}) ...")
    times = timeit.repeat(stmt=stmt, setup=setup, repeat=repeats, number=number)
    print("timeit results (seconds):", times)
    print(f"  mean: {statistics.mean(times):.2f}s, stdev: {statistics.stdev(times):.2f}s" if len(times) > 1 else f"  time: {times[0]:.2f}s")
    return times

def profile_sequential(run_download=False, profile_output="profile_stats.txt"):
    import cProfile, pstats, io
    pr = cProfile.Profile()
    pr.enable()
    sequential_pipeline_once(run_download=run_download)
    pr.disable()
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats("cumulative")
    ps.print_stats(40)
    out = s.getvalue()
    with open(profile_output, "w") as f:
        f.write(out)
    print(f"cProfile output written to {profile_output}")
    print("\n--- Top profiling lines (cumulative time) ---")
    print("\n".join(out.splitlines()[:40]))
    return out

# --------------------------------------------------
# MAIN + RESET GLOBALS + طباعة بعد الـ run الأساسي فقط
# --------------------------------------------------

if __name__ == "__main__":

    # نصفر القيم قبل التشغيل الأساسي
    GLOBAL_TOTAL_TOKENS = 0
    GLOBAL_BOOK_COUNTER = 0
    GLOBAL_TFIDF_SUM = 0.0

    print("Starting full (lock-based) pipeline...\n")
    results = sequential_pipeline_once(run_download=True)

    print_performance_report()

    # --- GLOBAL VALUES بعد الـ run الأساسي ---
    print("\n--- GLOBAL VALUES AFTER MAIN RUN (with Lock fix) ---")
    print("GLOBAL_TOTAL_TOKENS  :", GLOBAL_TOTAL_TOKENS)
    print("GLOBAL_BOOK_COUNTER  :", GLOBAL_BOOK_COUNTER)
    print("GLOBAL_TFIDF_SUM     :", GLOBAL_TFIDF_SUM)

    # --- TOP 15 TF-IDF + TOP 15 LEMMAS (المطلوب الأساسي) ---
    global_tfidf_agg = Counter()
    for book, scored in results["tfidf_per_book"].items():
        for term, score in scored:
            global_tfidf_agg[term] += score

    # لو عندك global_top_candidates في results تقدرين تكمّلينه كذا:
    for term in results["global_top_candidates"]:
        if term not in global_tfidf_agg:
            global_tfidf_agg[term] = 0.0

    top_15_tfidf = [t for t, _ in global_tfidf_agg.most_common(15)]

    print("\n--- TOP 15 MEANINGFUL WORDS (TF-IDF ranking across books) ---")
    for i, w in enumerate(top_15_tfidf, start=1):
        print(f"{i:2d}. {w}")

    print("\n--- TOP 15 MEANINGFUL LEMMAS (by frequency) ---")
    for i, (w, c) in enumerate(results["agg_uni"].most_common(15), start=1):
        print(f"{i:2d}. {w} : {c:,}")

    # نكتب التوب 15 في ملف مثل النسخة الأصلية
    with open("top_15_tfidf.txt", "w", encoding="utf-8") as f:
        for w in top_15_tfidf:
            f.write(w + "\n")
    print("\nSaved top_15_tfidf.txt and zipf_meaningful.png to current directory.")

    # بعدين لو حابة تشغّلين timeit / profile ما يأثرون على اللي فوق
    times = run_timeit_on_sequential(repeats=3, number=1)
    profile_text = profile_sequential(run_download=False, profile_output="profile_stats.txt")


Preloading WordNet before threading...


[nltk_data] Downloading package wordnet to /root/nltk_data...


WordNet is ready.

Starting full (lock-based) pipeline...

[stage start] download_books ...
[download] pride_and_prejudice ...
[download] moby_dick ...
[saved] gutenberg_books/pride_and_prejudice.txt (734.9 KB)
[download] war_and_peace ...
[saved] gutenberg_books/moby_dick.txt (1227.1 KB)
[download] ulysses ...
[saved] gutenberg_books/war_and_peace.txt (3280.7 KB)
[download] great_expectations ...
[saved] gutenberg_books/ulysses.txt (1530.0 KB)
[download] alice_in_wonderland ...
[saved] gutenberg_books/alice_in_wonderland.txt (147.6 KB)
[download] sherlock_holmes ...
[saved] gutenberg_books/great_expectations.txt (1014.2 KB)
[saved] gutenberg_books/sherlock_holmes.txt (593.3 KB)
[stage done]  download_books — 2.53s
[stage start] read_and_clean ...
  read: pride_and_prejudice — 711.7 KB
  read: moby_dick — 1190.4 KB
  read: war_and_peace — 3133.9 KB
  read: ulysses — 1484.1 KB
  read: alice_in_wonderland — 141.3 KB
  read: great_expectations — 971.4 KB
  read: sherlock_holmes — 549.9 KB

/usr/local/lib/python3.12/dist-packages/nltk/corpus/reader/wordnet.py:1580: UserWarning: No WordNet synset found for pos=v at offset=204872.
  warnings.warn(f"No WordNet synset found for pos={pos} at offset={offset}.")


  extracted meaningful lemmas for pride_and_prejudice: tokens=40,425, unique=4,400
  extracted meaningful lemmas for moby_dick: tokens=81,841, unique=10,269


/usr/local/lib/python3.12/dist-packages/nltk/corpus/reader/wordnet.py:1580: UserWarning: No WordNet synset found for pos=n at offset=10777768.
  warnings.warn(f"No WordNet synset found for pos={pos} at offset={offset}.")


  extracted meaningful lemmas for ulysses: tokens=98,850, unique=14,276
  extracted meaningful lemmas for alice_in_wonderland: tokens=8,432, unique=1,600
  extracted meaningful lemmas for war_and_peace: tokens=201,954, unique=10,350
  extracted meaningful lemmas for great_expectations: tokens=60,059, unique=7,172
  extracted meaningful lemmas for sherlock_holmes: tokens=34,751, unique=5,199
[stage done]  pos_lemmatize_meaningful — 161.00s
[stage start] ngram_counting ...
  ngrams for pride_and_prejudice: tokens=40,425, unique_unis=4,400, bigrams=35,952
  ngrams for moby_dick: tokens=81,841, unique_unis=10,269, bigrams=73,462
  ngrams for alice_in_wonderland: tokens=8,432, unique_unis=1,600, bigrams=7,259
  ngrams for ulysses: tokens=98,850, unique_unis=14,276, bigrams=89,012
  ngrams for great_expectations: tokens=60,059, unique_unis=7,172, bigrams=51,070
  ngrams for sherlock_holmes: tokens=34,751, unique_unis=5,199, bigrams=31,301
  ngrams for war_and_peace: tokens=201,954, unique_un